## WLC Mode Data Capture

This notebook demostrates how to enable wlc mode, and what it is used for.

### Naludaq Version
*Min Version*: `0.36.0`

### Compatible Boards
+ `ASoCv3`
+ `HDSoCv1_evalr2`
+ `HDSoCv2_evalr2`


## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import time
import numpy as np
from copy import deepcopy

from naludaq.board import Board
from naludaq.communication import AnalogRegisters, DigitalRegisters, ControlRegisters
from naludaq.parsers import get_parser
from naludaq.tools import get_pedestals_controller
from naludaq.tools.pedestals.pedestals_correcter import PedestalsCorrecter

from naludaq.backend import AcquisitionManager, RemoteAcquisition

from naluinstruments.Instruments import Siglent_FuncGen

import matplotlib.pyplot as plt

## Function Definitions

In [ ]:
def create_acquisition(name: str, board: Board, readout_metadata: dict = None) -> RemoteAcquisition:
    """Create an acquisition with the given name and optional readout metadata.
    Sets the output to the created acquisition and returns the acquisition object.
        Args:
            name (str): The name of the acquisition to be created.
            board (Board): The board object to be used for creating the acquisition.
            readout_metadata (dict, optional): A dictionary containing metadata for the readout. Defaults to None.
        Returns:
            RemoteAcquisition: The created acquisition object with the specified name and readout metadata.
    """
    am = AcquisitionManager(board)
    acquisition = am.create(name)
    acquisition.set_output()
    if readout_metadata:
        acquisition.readout_metadata = readout_metadata
    return acquisition

def parse_data(raw_data: dict, board: Board, correct_peds: bool = True) -> dict:
    """Parses an event dict that contains a 'rawdata' key.
        Args:
            raw_data (dict): A dictionary containing a 'rawdata' key with the raw data to be parsed.
            board (Board): The board object containing the parameters and pedestals for parsing.
            correct_peds (bool, optional): Whether to apply pedestal correction to the parsed event. Defaults to True.
        Returns:
            dict: A dictionary representing the parsed event.
    """
    PARSER = get_parser(board.params)
    evt = PARSER.parse(raw_data)
    if correct_peds:
        PC = PedestalsCorrecter(board.params, board.pedestals)
        PC.run(evt)
    return evt

def create_acq_name(base_name: str) -> str:
    """Create an acquisition name by appending a timestamp to the base name.
        Args:
            base_name (str): The base name to which the timestamp will be appended.
        Returns:
            str: The generated acquisition name with the timestamp in the form <base_name>_<timestamp>.
    """
    timestr = time.strftime("%Y-%m-%d-%H-%M-%S")
    return base_name + "_" + timestr

def clear_figs():
    """Clears all matplotlib figures"""
    plt.close("all")

def create_fig(figsize: tuple) -> plt.Figure:
    """Creates a matplotlib figure with the given size, constrained layout, and white background.
        Args:
            figsize (tuple): A tuple specifying the width and height of the figure.
        Returns:
            plt.Figure: A matplotlib figure object with the specified properties.
    """

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    fig.set_facecolor('white')
    return fig

def plot_single_event(event, show: bool = True):
    """Plots a single event to the current matplotlib figure. Each channel is plotted separately with markers and a legend.
        Args:
            event (dict): A dictionary representing the event to be plotted.
            show (bool, optional): Whether to display the plot. Defaults to True.
    """
    for chan, chan_data in enumerate(event["data"]):
        if isinstance(chan_data, np.ndarray) and chan_data.size == 0:
            continue
        if len(chan_data) == 0:
            continue
        if np.all(np.isnan(chan_data)):
            continue
        plt.plot(chan_data, '.-', markersize=10, label=f"Channel {chan}")
    plt.legend(fontsize=12)
    if show:
        plt.show()


## Directory Setup

In [ ]:
DATA_DIR = Path("./data").absolute()
DATA_DIR.mkdir(exist_ok=True)
SERVER_DIR = Path("./server").absolute()
SERVER_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("figures").absolute()
FIGURES_DIR.mkdir(exist_ok=True)

## Board Initialization

In [ ]:
BOARD = Board("hdsocv2_evalr2")
BOARD.start_server(str(SERVER_DIR))
BOARD_ADDRESS = ("192.168.1.60", 4660)
RECEIVER_ADDRESS = ("192.168.1.165", 4665)
SERVER_ADDRESS = (BOARD.context.address)
BOARD.connect_udp(BOARD_ADDRESS, RECEIVER_ADDRESS, SERVER_ADDRESS)
BOARD.initialize()

## Signal Generator

A burst of pulses is sent into the board's `Trigger In`. In normal operation (wlcon=0), the chip will have a longer dead time when determining when a trigger will yield an event which depends on each chip. With WLC mode operation, this dead time will be reduced down to as low as ~100ns (depends on chip/configuration). There can be dead time if enough windows are exhausted for digitization in a short period of time.

We use a signal generator's burst mode, to send in a burst of N pulses to test if the board is able to trigger N events in short succession.

### Signal Generator Settings

In [ ]:
# A signal generator output going into a single channel of the board. We use a sinewave to confirm that non-contiguous windows are being read out correctly.
# 11 MHz sine wave, 100 mV amplitude, 50 ohm load, no burst
SINE_FREQ = 11e6
SINE_AMP = 0.1

# Pulse settings for the trigger going into the board's Trigger In. Use your corresponding board's logic level for the amplitude.
# The pulse frequency here is used to set the time between each pulse within a burst.
# Pulse Waveform, 5 microsecond pulse period, 50 ns pulse width, 2.5 V amplitude, 50 ohm load,
# 3 pulses per burst, 0.5 second burst period, internal trigger source
PULSE_WIDTH = 50e-9
PULSE_FREQ = 1 / (5e-6)
N_CYCLES = 3
BURST_PERIOD = 0.5

### Signal Generator Setup

In [ ]:
# This is using NaluInstruments which is internal to Nalu Scientific.
# Use the appropriate instrument control code for your signal generator.
CHANNEL_INPUT = Siglent_FuncGen(1, "192.168.1.124")
TRIGGER_INPUT = Siglent_FuncGen(2, "192.168.1.124")
CHANNEL_INPUT.rf_off()
TRIGGER_INPUT.rf_off()


CHANNEL_INPUT.set_burst("OFF")
CHANNEL_INPUT.set_load("50")
CHANNEL_INPUT.set_wavetype("SINE")
CHANNEL_INPUT.set_offset(0)
CHANNEL_INPUT.set_freq(SINE_FREQ)
CHANNEL_INPUT.set_amp(SINE_AMP)

TRIGGER_INPUT.set_burst("OFF")
TRIGGER_INPUT.set_load("50")
TRIGGER_INPUT.set_wavetype("PULSE")
TRIGGER_INPUT.set_offset(0)
TRIGGER_INPUT.set_freq(PULSE_FREQ)
TRIGGER_INPUT.set_amp(2.5)
TRIGGER_INPUT.set_width(PULSE_WIDTH)
TRIGGER_INPUT.set_burst("ON")
TRIGGER_INPUT.set_burst_mode("NCYC")
TRIGGER_INPUT.set_burst_time(N_CYCLES)
TRIGGER_INPUT.set_burst_period(BURST_PERIOD)
TRIGGER_INPUT.set_delay(0)
TRIGGER_INPUT.set_burst_triggersource("MAN")

print(CHANNEL_INPUT.get_output_settings())
print(CHANNEL_INPUT.get_burst_settings())
print(TRIGGER_INPUT.get_output_settings())
print(TRIGGER_INPUT.get_burst_settings())

## Data Capture Configuration

In [ ]:
DAC_LEVEL = 1804 # (Default DAC Counts for HDSoCv2) Adjust depending on your board.
WINDOWS = 8
LOOKBACK = WINDOWS + 2
WAT = 0
READOUT_CHANNELS = [0, 1, 2, 3]
WLC_ON = 1
READOUT_TIME = 5

# External Trigger, Trigger Relative Mode
READOUT_PARAMS = {
    "trig": "ext",
    "lb": "trigrel",
}

READ_WINDOW = {
    "windows": WINDOWS,
    "lookback": LOOKBACK,
    "write_after_trig": WAT,
}

# Created to save with the acquisition metadata for reference when analyzing the data and plotting.
PARAMS = {
    "dac_level": DAC_LEVEL,
    "windows": WINDOWS,
    "lookback": LOOKBACK,
    "write_after_trig": WAT,
    "n_cycle": N_CYCLES,
    "burst_period": BURST_PERIOD,
    "pulse_width": PULSE_WIDTH,
    "pulse_freq": PULSE_FREQ,
    "readout_time": READOUT_TIME,
    "wlc_on": WLC_ON,
    "readout_channels": READOUT_CHANNELS,
}

AR = AnalogRegisters(BOARD)
DR = DigitalRegisters(BOARD)
AM = AcquisitionManager(BOARD)

### Set Board Configurations

In [ ]:
BOARD.ext_bias.set_dacs(DAC_LEVEL)

### Capture Pedestals
The board is initialized before capturing pedestals in case WLC mode is still enabled. Pedestals require normal operation in order to capture the needed windows.

In [ ]:
PG = get_pedestals_controller(BOARD, num_captures=10, num_warmup_events=10)
# Use normal operation (Disable WLC mode)
DR.write("wlc_on", 0)
BOARD.initialize()
PG.generate_pedestals()

### Set Readout Settings
Similarly to pedestals, if normal mode is being used \(`WLC_ON=0`\), we initialize the board in case it was in WLC mode.

In [ ]:
# Enable WLC mode
DR.write("wlc_on", WLC_ON)
# This initialize is needed for a Normal Operation (WLC_ON=0) readout, if WLC was enabled (WLC_ON=1) prior.
# Can be skipped otherwise.
BOARD.initialize()

BOARD.readout.set_read_window(**READ_WINDOW)
BOARD.readout.set_readout_channels(READOUT_CHANNELS)

### Create an Acquisition

In [ ]:
ACQ_NAME = f"{BOARD.model}_windows{WINDOWS}_ncyc{N_CYCLES}_wlc{WLC_ON}"
acq_name = create_acq_name(ACQ_NAME)
readout_metadata = {"params": deepcopy(PARAMS)}
acq = create_acquisition(acq_name, BOARD, readout_metadata)
acq.pedestals = BOARD.pedestals

## Readout Data

### Start Readout

In [ ]:
try:
    BOARD.control.start_readout(**READOUT_PARAMS)
    time.sleep(0.1)
    CHANNEL_INPUT.rf_on()
    TRIGGER_INPUT.rf_on()
except:
    BOARD.control.stop_readout()
    TRIGGER_INPUT.rf_off()
    CHANNEL_INPUT.rf_off()
    AM.current_acquisition = None

### Manually Pulse In Triggers

In [ ]:
TRIGGER_INPUT.set_burst_mtrig(1)

### Stop Readout

In [ ]:
BOARD.control.stop_readout()
TRIGGER_INPUT.rf_off()
CHANNEL_INPUT.rf_off()
time.sleep(0.5)
print(acq.length)

## Parse and Plot Events

In [ ]:
PARSER = get_parser(BOARD.params)

# acq = AM.list()[0] # Get the first named acq from the backend
print(acq.readout_metadata)
event_num = 0
parsed = acq.parsed_event(PARSER, index=event_num)

# ASOCv3 only: Events return a "WLC Header", which indicates the order in which the WLC windows were read out. This is important for correctly plotting the data.
# parsed["wlc_headers"]

fig = create_fig((10, 6))
plot_single_event(parsed, show=False)
plt.savefig(FIGURES_DIR / f"{acq.name}_event{event_num}.png")